# Track B, step 8: LoRA-149M text scores for the final model

**Why this step:** `trackb_7_lora_granite_149m.ipynb` showed LoRA-149M passed the quick check on 2025. To be used **inside** the final model, the text score has to exist for every row the model trains on, and each of those scores must come from a text model that **never saw that row's label**. Otherwise the final model would learn to trust a text score that has memorized the answers.

**The files, same layout as the frozen-Granite set** (`trackb_5_frozen_handoff.ipynb`), saved to `data/trackb/handoff_lora149/`:

| File | Text model trained on | Rows scored | Used for |
|---|---|---|---|
| `text_oof_train.csv` | 5 folds of 2018-2024 (labels by Dec 31, 2024) | 2024 backtest rows, out-of-fold | the combination test (train on 2024) |
| `text_val_2025.csv` | all of 2018-2024 | 2025 rows | the combination test (test on 2025); **already saved by step 7** |
| `text_oof_final.csv` | 5 folds of 2018-2025 (labels by Dec 31, 2025) | 2024-2025 backtest rows, out-of-fold | training the final model |
| `text_2026.csv` | all of 2018-2025 | all 72,670 scoreboard rows | scoring 2026 |

**Settings:** exactly those of step 7 (`ml/config.py`, `LORA`), unchanged. Each file is saved as soon as it's done, and a file that already exists is not recomputed, so the notebook can resume after an interruption.

It runs 11 LoRA trainings (5 + 5 + 1) on the Mac's GPU: several hours.

In [ ]:
import sys, time, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = next(p for p in [Path(".."), Path(".")] if (p / "ml" / "config.py").exists()).resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
from ml import config, text_model as T

OUT = config.TEXT_SCORES["lora-149m"]; OUT.mkdir(parents=True, exist_ok=True)
texts = T.load_texts()
bt24 = pd.read_parquet(config.DATA_DIR / "backtest/train_2024.parquet", columns=["cve_id"]).cve_id
bt25 = pd.read_parquet(config.DATA_DIR / "backtest/val_2025.parquet", columns=["cve_id"]).cve_id
tr = T.labeled(texts, "train", "2024-12-31")                 # 2018-2024
both = T.labeled(texts, ["train", "val"], "2025-12-31")      # 2018-2025
s26 = texts[texts.split == "score26"].sort_values("cve_id", ignore_index=True)
mask24 = tr.cve_id.isin(bt24).to_numpy()
mask_final = both.cve_id.isin(pd.concat([bt24, bt25])).to_numpy()
assert mask24.sum() == len(bt24) and mask_final.sum() == len(bt24) + len(bt25) and len(s26) == 72_670
print(f"device {T.device()} | 2018-2024: {len(tr):,} rows, {tr.y.sum()} positives -> score {mask24.sum():,} | "
      f"2018-2025: {len(both):,} rows, {both.y.sum()} positives -> score {mask_final.sum():,} | 2026: {len(s26):,}")
LOGS = []
def step(name, make):
    f = OUT / name
    if f.exists():
        print(f"{name}: already saved, skipped"); return pd.read_csv(f)
    t0 = time.time(); df, log = make()
    assert df.cve_id.is_unique and df.text_score.notna().all()
    df.to_csv(f, index=False); LOGS.append(log.assign(file=name))
    print(f"{name}: {len(df):,} rows in {(time.time() - t0) / 60:.0f} min"); return df

## 1. Out-of-fold over 2018-2024 (for the combination test)

In [ ]:
oof_train = step("text_oof_train.csv", lambda: T.out_of_fold(tr, mask24, label="2018-2024"))

## 2. Out-of-fold over 2018-2025 (for the final model)

In [ ]:
oof_final = step("text_oof_final.csv", lambda: T.out_of_fold(both, mask_final, label="2018-2025"))

## 3. One model on all of 2018-2025, scoring 2026

In [ ]:
def score_2026():
    model, log = T.fit(both, label="2018-2025, all")
    T._free_memory()
    s = T.predict(model, s26)
    model.save_pretrained(config.TEXT_DIR / "lora149_adapter_2018_2025")
    return pd.DataFrame({"cve_id": s26.cve_id, "text_score": s}), log
text_2026 = step("text_2026.csv", score_2026)

## 4. Checks

- **All four files exist**, with the expected row counts.
- **Out-of-fold scores are informative but not suspiciously perfect.** If a text model had seen the labels, its AUC on the training rows would be near 1.0. An honest out-of-fold AUC should look like the 2025 validation AUC.

In [ ]:
val = pd.read_csv(OUT / "text_val_2025.csv")
expected = {"text_oof_train.csv": mask24.sum(), "text_val_2025.csv": 35_281, "text_oof_final.csv": mask_final.sum(), "text_2026.csv": 72_670}
for name, n in expected.items():
    d = pd.read_csv(OUT / name); assert len(d) == n and d.cve_id.is_unique and d.text_score.notna().all(), name
print("four files, row counts, unique IDs, no blanks: OK")
def auc(scores, rows):
    m = rows[["cve_id", "y"]].merge(scores, on="cve_id"); return roc_auc_score(m.y, m.text_score), int(m.y.sum())
for name, sc, rows in [("out-of-fold 2024 (2018-2024 folds)", oof_train, tr[mask24]),
                       ("validation 2025 (trained on 2018-2024)", val, T.labeled(texts, "val", "2025-12-31")),
                       ("out-of-fold 2024-2025 (2018-2025 folds)", oof_final, both[mask_final])]:
    a, n = auc(sc, rows); print(f"{name}: AUC {a:.3f} ({n} positives)")
if LOGS:
    pd.concat(LOGS, ignore_index=True).to_csv(OUT / "training_log_handoff.csv", index=False)
    display(pd.concat(LOGS, ignore_index=True))